In [11]:
!wget https://github.com/shenwei356/seqkit/releases/download/v2.13.0/seqkit_linux_amd64.tar.gz
!tar -xzf seqkit_linux_amd64.tar.gz
!mv seqkit /usr/local/bin/
!pip install biopython

import os
import requests
import re
import json
from Bio import SeqIO
import subprocess
import sys

--2026-03-25 19:42:07--  https://github.com/shenwei356/seqkit/releases/download/v2.13.0/seqkit_linux_amd64.tar.gz
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/52715040/02fe855c-526a-4cff-805c-975f91267b88?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-25T20%3A19%3A09Z&rscd=attachment%3B+filename%3Dseqkit_linux_amd64.tar.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-25T19%3A18%3A14Z&ske=2026-03-25T20%3A19%3A09Z&sks=b&skv=2018-11-09&sig=CtTLZhzXeN9JuuK%2FwyJnHbz6yTqvTbNsVGWZibasvcA%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3NDQ2ODAyNywibmJmIjoxNzc0NDY3NzI3LCJwYXRoIjoicmVsZWFzZWFzc2

In [22]:
class MyFastaParser:
    def __init__(self, file_name):
        self.filename = file_name

    def _get_uniprot(self, accession):
        url = f'https://rest.uniprot.org/uniprotkb/{accession}'
        return requests.get(url, headers={'Accept': 'application/json'})

    def _get_ensembl(self, id):
        url = f'https://rest.ensembl.org/lookup/id/{id}?content-type=application/json'
        return requests.get(url)

    def _uniprot_parse_response(self, resp: dict):
        data = resp.json() if hasattr(resp, 'json') else resp
        organism = data.get('organism', {}).get('scientificName')
        geneInfo = data.get('genes', [])
        sequenceInfo = data.get('sequence', {})
        output = {
            'organism': organism,
            'geneInfo': geneInfo,
            'sequenceInfo': sequenceInfo,
            'type': 'protein'
        }
        return output

    def _ensembl_parse_response(self, resp: dict):
        data = resp.json() if hasattr(resp, 'json') else resp
        keys_to_extract = [
            'object_type', 'assembly_name', 'species', 'db_type', 'biotype',
            'display_name', 'id', 'description', 'canonical_transcript', 'source'
        ]
        output = {k: data.get(k) for k in keys_to_extract if k in data}
        return output

    def _access_database(self, id, database, seq_description, seq_sequence) -> dict:
        if database == 'uniprot':
            resp = self._get_uniprot(id)
            db_info = self._uniprot_parse_response(resp) if resp.status_code == 200 else {'error': 'API Error'}
        else:
            resp = self._get_ensembl(id)
            db_info = self._ensembl_parse_response(resp) if resp.status_code == 200 else {'error': 'API Error'}

        output = {
            'DB_name': database,
            'file_info_' + id: {
                'description': seq_description,
                'sequence': str(seq_sequence)
            },
            'database_info_' + id: db_info
        }
        return output

    def seqkit_stats(self) -> dict:
        try:
            result = subprocess.run(
                f'seqkit stats -a -T {self.filename}',
                capture_output=True,
                text=True,
                check=True,
                shell=True
            )
            lines = result.stdout.strip().split('\n')
            if len(lines) < 2:
                return {'error': result.stderr.strip()}
            headers = lines[0].split('\t')
            values = lines[1].split('\t')
            stat_info = dict(zip(headers, values))
            return {
                'fasta_seqkit_stat_info': stat_info,
                'fasta_type': stat_info.get('type', ''),
                'fasta_num_seqs': int(stat_info.get('num_seqs', 0))
            }
        except subprocess.CalledProcessError as e:
            return {'error': e.stderr}
        except FileNotFoundError:
            return {'error': 'Seqkit not found. Ensure it is installed and in PATH.'}

    def biopython_parser(self, seqkit_result) -> dict:
        if 'error' in seqkit_result:
            return seqkit_result

        seq_type = seqkit_result.get('fasta_type', '')

        if seq_type == 'Protein':
            db_name = 'uniprot'
            pattern = re.compile(r'[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}')
        elif seq_type in ['DNA', 'RNA']:
            db_name = 'ensembl'
            pattern = re.compile(r'ENS[A-Z]*[0-9]{11}')
        else:
            return {'error': f'Not supported sequence type: {seq_type}'}

        # privet prof i hope you are doing well ;)
        ultimate_output = {'DB_name': db_name}
        warnings = set()

        for record in SeqIO.parse(self.filename, 'fasta'):
            description = record.description
            match = pattern.search(description)
            if match:
                found_id = match.group(0)
                ultimate_output['file_info_' + found_id] = {
                    'description': description,
                    'sequence': str(record.seq)
                }
                if db_name == 'uniprot':
                    resp = self._get_uniprot(found_id)
                    db_data = self._uniprot_parse_response(resp) if resp.status_code == 200 else {'error': 'API Error'}
                else:
                    resp = self._get_ensembl(found_id)
                    db_data = self._ensembl_parse_response(resp) if resp.status_code == 200 else {'error': 'API Error'}
                ultimate_output['database_info_' + found_id] = db_data
            else:
                warnings.add('No ID match found.')

        if warnings:
            ultimate_output['WARNING'] = warnings

        return ultimate_output

    def show_output(self, output, indent=0):
        for key, value in output.items():
            print('\t' * indent + str(key))
            if isinstance(value, dict):
                self.show_output(value, indent + 1)
            else:
                print('\t' * (indent + 1) + str(value))

In [23]:
parser = MyFastaParser('test_file.fasta')
stats = parser.seqkit_stats()
print(json.dumps(stats, indent=2))

{
  "fasta_seqkit_stat_info": {
    "file": "test_file.fasta",
    "format": "FASTA",
    "type": "Protein",
    "num_seqs": "2",
    "sum_len": "456",
    "min_len": "29",
    "avg_len": "228.0",
    "max_len": "427",
    "Q1": "29",
    "Q2": "228",
    "Q3": "427",
    "sum_gap": "0",
    "N50": "427",
    "N50_num": "1",
    "Q20(%)": "0",
    "Q30(%)": "0",
    "AvgQual": "0.00",
    "GC(%)": "0.00",
    "sum_n": "0"
  },
  "fasta_type": "Protein",
  "fasta_num_seqs": 2
}


In [24]:
biopython = parser.biopython_parser(stats)
parser.show_output(biopython)

DB_name
	uniprot
file_info_P11473
	description
		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
	sequence
		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
database_info_P11473
	organism
		Homo sapiens
	geneInfo
		[{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312', 'source': 'HGNC', 'id': 'HGNC:12679'}], 'value': 'VDR'}, 'synonyms': [{'value': 'NR1I1'}]}]
	sequenceInfo
		value
			MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSS

In [25]:
parser_1 = MyFastaParser('ensembl_download_1.fasta')
stats_1 = parser_1.seqkit_stats()
print(json.dumps(stats_1, indent=2))

{
  "fasta_seqkit_stat_info": {
    "file": "ensembl_download_1.fasta",
    "format": "FASTA",
    "type": "DNA",
    "num_seqs": "6",
    "sum_len": "86",
    "min_len": "9",
    "avg_len": "14.3",
    "max_len": "23",
    "Q1": "10",
    "Q2": "14",
    "Q3": "17",
    "sum_gap": "0",
    "N50": "16",
    "N50_num": "3",
    "Q20(%)": "0",
    "Q30(%)": "0",
    "AvgQual": "0.00",
    "GC(%)": "45.35",
    "sum_n": "0"
  },
  "fasta_type": "DNA",
  "fasta_num_seqs": 6
}


In [26]:
biopython_1 = parser_1.biopython_parser(stats_1)
parser_1.show_output(biopython_1)

DB_name
	ensembl
file_info_ENSMUST00000196221
	description
		ENSMUST00000196221.2 cds chromosome:GRCm39:14:54350925:54350933:1 gene:ENSMUSG00000096749.3 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd1 description:T cell receptor delta diversity 1 [Source:MGI Symbol;Acc:MGI:4439547]
	sequence
		ATGGCATAT
database_info_ENSMUST00000196221
	object_type
		Transcript
	assembly_name
		GRCm39
	species
		mus_musculus
	db_type
		core
	biotype
		TR_D_gene
	display_name
		Trdd1-202
	id
		ENSMUST00000196221
	source
		havana
file_info_ENSMUST00000177564
	description
		ENSMUST00000177564.2 cds chromosome:GRCm39:14:54359683:54359698:1 gene:ENSMUSG00000096176.2 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd2 description:T cell receptor delta diversity 2 [Source:MGI Symbol;Acc:MGI:4439546]
	sequence
		ATCGGAGGGATACGAG
database_info_ENSMUST00000177564
	object_type
		Transcript
	assembly_name
		GRCm39
	species
		mus_musculus
	db_type
		core
	biotype
		TR_D_gene


In [27]:
parser_2 = MyFastaParser('ensembl_download_2.fasta')
stats_2 = parser_2.seqkit_stats()
print(json.dumps(stats_2, indent=2))

{
  "error": "\u001b[ERRO]\u001b ensembl_download_2.fasta: fastx: invalid FASTA/Q format"
}


In [28]:
biopython_2 = parser_2.biopython_parser(stats_2)
parser_2.show_output(biopython_2)

error
	[ERRO] ensembl_download_2.fasta: fastx: invalid FASTA/Q format


In [29]:
parser_3 = MyFastaParser('uniprot_download.fasta')
stats_3 = parser_3.seqkit_stats()
print(json.dumps(stats_3, indent=2))

{
  "fasta_seqkit_stat_info": {
    "file": "uniprot_download.fasta",
    "format": "FASTA",
    "type": "Protein",
    "num_seqs": "7",
    "sum_len": "3861",
    "min_len": "180",
    "avg_len": "551.6",
    "max_len": "1382",
    "Q1": "429",
    "Q2": "441",
    "Q3": "500",
    "sum_gap": "0",
    "N50": "468",
    "N50_num": "3",
    "Q20(%)": "0",
    "Q30(%)": "0",
    "AvgQual": "0.00",
    "GC(%)": "0.00",
    "sum_n": "0"
  },
  "fasta_type": "Protein",
  "fasta_num_seqs": 7
}


In [30]:
biopython_3 = parser_3.biopython_parser(stats_3)
parser_3.show_output(biopython_3)

DB_name
	uniprot
file_info_Q9R1A7
	description
		sp|Q9R1A7|NR1I2_RAT Nuclear receptor subfamily 1 group I member 2 OS=Rattus norvegicus OX=10116 GN=Nr1i2 PE=2 SV=1
	sequence
		MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLLPHLADVSTYMFKGVINFAKVISHFRELPIEDQISLLKGATFEMCILRFNTMFDTETGTWECGRLAYCFEDPNGGFQKLLLDPLMKFHCMLKKLQLREEEYVLMQAISLFSPDRPGVVQRSVVDQLQERFALTLKAYIECSRPYPAHRFLFLKIMAVLTELRSINAQQTQQLLRIQDTHPFATPLMQELFSSTDG
database_info_Q9R1A7
	organism
		Rattus norvegicus
	geneInfo
		[{'geneName': {'value': 'Nr1i2'}, 'synonyms': [{'value': 'Pxr'}]}]
	sequenceInfo
		value
			MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLL